# Test Experement

In [1]:
import sys
import os
from pathlib import Path
from typing import List

# Add project root to path (adjust if notebook is in a subfolder)
project_root = Path.cwd().parent  # if notebook is in experiments/ or similar
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import warnings

# Suppress the specific future warning from torchrl
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    module="torchrl.modules.mcts.scores"
)

import torch

# BenchMARL
from benchmarl.algorithms import (
    # full observation in critic
    MappoConfig,
    MaddpgConfig,
    MasacConfig, 
    # no full observation in critic
    IppoConfig,
    IddpgConfig,
    IsacConfig,
    # Discrete only
    #QmixConfig
)
from benchmarl.benchmark import Benchmark
# from benchmarl.environments import VmasTask
from benchmarl.experiment import Experiment, ExperimentConfig
from benchmarl.models.mlp import MlpConfig
from benchmarl.eval_results import load_and_merge_json_dicts, Plotting

# VUMAS task for BenchMARL
from benchmarl.environments import UrbanEnvTask


# Ploting
from matplotlib import pyplot as plt

# Main Configuration Parameters

In [2]:
# Senario
senario = "uav_navigation"

# Experiment parameters
num_envs = 72
max_n_steps = 200
max_n_iters = 20

frames_per_batch = int(num_envs * max_n_steps)
max_n_frames = int(frames_per_batch * max_n_iters)
# test_los: has the following conf
# max_horizontal_speed: 490
# max_vertical_speed: 12.0
# output_dir = project_root / "outputs" / "test_los"

# test_los: has the following conf
# max_horizontal_speed: 500.0
# max_vertical_speed: 50.0
output_dir = project_root / "outputs" / f"{senario}_1"

## Configure Experement

In [3]:
# Configure Experement
# Loads from "benchmarl/conf/experiment/base_experiment.yaml"
experiment_config = ExperimentConfig.get_from_yaml()
# configure experiment
if torch.cuda.is_available():
    experiment_config.device = "cuda"
    experiment_config.sampling_device = "cuda"
    experiment_config.train_device = "cuda"
    experiment_config.buffer_device = "cuda"


# Whether to share the parameters of the policy within agent groups
# experiment_config.share_policy_params = True
# If an algorithm and an env support both continuous and discrete actions, what should be preferred
# experiment_config.prefer_continuous_actions = True
# If False collection is done using a collector (under no grad). If True, collection is done with gradients.
# experiment_config.collect_with_grad = False
# In case of non-vectorized environments, whether to run collection of multiple processes
# If this is used, there will be n_envs_per_worker processes, collecting frames_per_batch/n_envs_per_worker frames each
experiment_config.parallel_collection = True # False

# Discount factor
# experiment_config.gamma = 0.99
# Learning rate
# experiment_config.lr = 0.00005
# The epsilon parameter of the adam optimizer
# experiment_config.adam_eps = 0.000001
# Extra kwargs for the adam optimizer
# experiment_config.adam_extra_kwargs = {}
# Clips grad norm if true and clips grad value if false
# experiment_config.clip_grad_norm = True
# The value for the clipping, if null no clipping
# experiment_config.clip_grad_val = 5

# Whether to use soft or hard target updates
# experiment_config.soft_target_update = True
# If soft_target_update is True, this is its polyak_tau
# experiment_config.polyak_tau = 0.005
# If soft_target_update is False, this is the frequency of the hard target updates in terms of n_optimizer_steps
# experiment_config.hard_target_update_frequency = 5

# When an exploration wrapper is used. This is its initial epsilon for annealing
# experiment_config.exploration_eps_init = 0.8
# When an exploration wrapper is used. This is its final epsilon after annealing
# experiment_config.exploration_eps_end = 0.01
# Number of frames for annealing of exploration strategy in deterministic policy algorithms
# If null it will default to max_n_frames / 3
# experiment_config.exploration_anneal_frames = null

# The maximum number of experiment iterations before the experiment terminates, exclusive with max_n_frames
experiment_config.max_n_iters = max_n_iters
# Number of collected frames before ending, exclusive with max_n_iters
experiment_config.max_n_frames = max_n_frames

# Number of frames collected and each experiment iteration
experiment_config.on_policy_collected_frames_per_batch = int(num_envs * max_n_steps)
# -------------------------------
# Number of environments used for collection
# If the environment is vectorized, this will be the number of batched environments.
# Otherwise batching will be simulated and each env will be run sequentially or parallelly depending on parallel_collection.
experiment_config.on_policy_n_envs_per_worker = num_envs
# -------------------------------
# This is the number of times collected_frames_per_batch will be split into minibatches and trained
# experiment_config.on_policy_n_minibatch_iters = 45
# In on-policy algorithms the train_batch_size will be equal to the on_policy_collected_frames_per_batch
# and it will be split into minibatches with this number of frames for training
# experiment_config.on_policy_minibatch_size = 400

# Number of frames collected and each experiment iteration
experiment_config.off_policy_collected_frames_per_batch = int(num_envs * max_n_steps)
# -------------------------------
# Number of environments used for collection
# If the environment is vectorized, this will be the number of batched environments.
# Otherwise batching will be simulated and each env will be run sequentially or parallelly depending on parallel_collection.
experiment_config.off_policy_n_envs_per_worker = num_envs
# -------------------------------
# This is the number of times off_policy_train_batch_size will be sampled from the buffer and trained over.
# experiment_config.off_policy_n_optimizer_steps = 1000
# Number of frames used for each off_policy_n_optimizer_steps when training off-policy algorithms
# experiment_config.off_policy_train_batch_size = 128
# Maximum number of frames to keep in replay buffer memory for off-policy algorithms
# experiment_config.off_policy_memory_size = 1_000_000
# Number of random action frames to prefill the replay buffer with
# experiment_config.off_policy_init_random_frames = 0
# whether to use priorities while sampling from the replay buffer
# experiment_config.off_policy_use_prioritized_replay_buffer = False
# exponent that determines how much prioritization is used when off_policy_use_prioritized_replay_buffer = True
# PRB reduces to random sampling when alpha=0
# experiment_config.off_policy_prb_alpha = 0.6
# importance sampling negative exponent when off_policy_use_prioritized_replay_buffer = True
# experiment_config.off_policy_prb_beta = 0.4


# experiment_config.evaluation = True
# Whether to render the evaluation (if rendering is available)
# experiment_config.render = True
# Frequency of evaluation in terms of collected frames (this should be a multiple of on/off_policy_collected_frames_per_batch)
experiment_config.evaluation_interval = int(2 * num_envs * max_n_steps)
# Number of episodes that evaluation is run on
# experiment_config.evaluation_episodes = 10
# If True, when stochastic policies are evaluated, their deterministic value is taken, otherwise, if False, they are sampled
# experiment_config.evaluation_deterministic_actions = True
# If True, seed the environment before evaluation leading to always the same evaluation env being used
# If False, evaluation environments will be more random throughout training
# experiment_config.evaluation_static = False

# List of loggers to use, options are = wandb, csv, tensorboard, mflow
# experiment_config.loggers = [tensorboard]
# Wandb project name (kept for backward compatibility)
# experiment_config.project_name = "benchmarl"
# Wandb extra kwargs passed to the WandbLogger (~superset of wandb.init kwargs)
# WandbLogger includes = offline, save_dir, project, video_fps
# wandb.init includes = entity, tags, notes, etc.
# experiment_config.wandb_extra_kwargs = {}
# Create a json folder as part of the output in the format of marl-eval
# experiment_config.create_json = True

# Absolute path to the folder where the experiment will log.
# If null, this will default to the hydra output dir (if using hydra) or to the current folder when the script is run (if not).
# If you are reloading an experiment with "restore_file", this will default to the reloaded experiment folder.
experiment_config.save_folder = output_dir

# Absolute path to a checkpoint file where the experiment was saved. If null the experiment is started fresh.
# experiment_config.restore_file = null

# Map location given to `torch.load()` when reloading.
# If you are reloading in a cpu-only machine a gpu experiment, you can use `restore_map_location = {"cuda:0":"cpu"}`
# to map gpu tensors to the cpu
# experiment_config.restore_map_location = null

# Interval for experiment saving in terms of collected frames (this should be a multiple of on/off_policy_collected_frames_per_batch).
# Set it to 0 to disable checkpointing
experiment_config.checkpoint_interval = 0 #int(2 * num_envs * max_n_steps)
# Whether to checkpoint when the experiment is done
experiment_config.checkpoint_at_end = False # Treu
# How many checkpoints to keep. As new checkpoints are taken, temporally older checkpoints are deleted to keep this number of
# checkpoints. The checkpoint at the end is included in this number. Set to `null` to keep all checkpoints.
# experiment_config.keep_checkpoints_num = 3
# Whether to exclude the replay buffers from the checkpoint
# experiment_config.exclude_buffer_from_checkpoint = False

# -------------
os.makedirs(experiment_config.save_folder, exist_ok=True)

In [6]:
# Loads from "benchmarl/conf/task/urbanmarl"
tasks = [
    # UrbanEnvTask.UAV_UE_LOS.get_from_yaml(),
    UrbanEnvTask.UAV_NAVIGATION.get_from_yaml(),
    # UrbanEnvTask.UAVMEC_OFFLOADING.get_from_yaml(),
    # UrbanEnvTask.COVERAGE.get_from_yaml(),
]

# Loads from "benchmarl/conf/model/layers"
model_config = MlpConfig.get_from_yaml()
critic_model_config = MlpConfig.get_from_yaml()



In [7]:
# BenchMARL algorithms
# from benchmarl.algorithms import (
#     # full observation in critic
#     MappoConfig,
#     MaddpgConfig,
#     MasacConfig, 
#     # no full observation in critic
#     IppoConfig,
#     IddpgConfig,
#     IsacConfig,
#     # Discrete only
#     #QmixConfig
# )

_algorithm_configs = [
    # On-policy algorithms
    MappoConfig.get_from_yaml(),
    MaddpgConfig.get_from_yaml(),
    MasacConfig.get_from_yaml(),
    
    # Off-policy algorithms
    IppoConfig.get_from_yaml(),
    IddpgConfig.get_from_yaml(),
    IsacConfig.get_from_yaml(),
]

In [ ]:
task = tasks[0]
algorithm_config = _algorithm_configs[0]
for seed in [2, 3, 4]:
    for algorithm_config in _algorithm_configs:
        experiment = Experiment(
            task=task,
            algorithm_config=algorithm_config,
            model_config=model_config,
            critic_model_config=critic_model_config,
            seed=seed,
            config=experiment_config,
            # callbacks=[EvaluateLoS()],
        )
        experiment.run()

In [11]:
experiment = Experiment(
            task=tasks[0],
            algorithm_config=_algorithm_configs[0],
            model_config=model_config,
            critic_model_config=critic_model_config,
            seed=0,
            config=experiment_config,
            # callbacks=[EvaluateLoS()],
        )

/doc/code/papers/DRL/vUrbanMARL/benchmarl/experiment/experiment.py:314: UserWarning: max_n_frames and max_n_iters have both been set. The experiment will terminate after 20 iterations (288000 frames).
  warnings.warn(
/home/yemenlinux/miniconda3/envs/urbanmarl/lib/python3.12/site-packages/tensordict/_td.py:612: FutureWarning: TensorDict.to_module() is replacing an existing nn.Parameter in the destination module with a tensor leaf that is not an nn.Parameter. This historical behavior can remove the key from module.state_dict(). In tensordict v0.14, to_module() will preserve existing module parameter and buffer registrations by default. Pass preserve_module_state=False to keep the current replacement behavior, or preserve_module_state=True to opt in to the v0.14 behavior now.
  local_out = _set_tensor_dict(


In [12]:
experiment.algorithm_name

'mappo'